# Taux de rapportage des données de routine

Ce rapport génère des visualisations liées aux taux de rapportage des principaux indicateurs de routine, au niveau administratif ADM2. Le taux de rapportage permet évaluer l'exhaustivité des données de routine et les cas où la fiabilité des indicateurs utilisés pourrait compromettre la prise de décision.

Le rapport se base sur les valeurs des paramètres utilisées dans le notebook de calcul de des taux de rapportage, faisant partie du même pipeline. Il s'agit des choix suivants:

* La modalité de déterminer quelles FOSA sont censées envoyer des rapports:
    - FOSA ouvertes
    - FOSA actives
* Les indicateurs choisis pour déterminer l'état d'activité des FOSA
* Les indicateurs choisis pour déterminer le volume d'activité des FOSA
* Type de données de routine utilisées :
    - Concernant les cas extrêmes, il s'agira d'un des choix suivants :
        - brutes
        - sans cas extrêmes
        - avec les cas extrêmes imputés
    - Concernant les pondérations, il s'agira d'un des choix suivants:
        - brutes
        - pondérées par le volume d'activité

Le rapport produit notamment les graphiques suivants :

* Nuage de points du taux de rapportage au niveau administratif ADM2, par année et mois
* Carte thermique mensuelle du taux de rapportage moyen
* Carte du taux de rapportage par mois et par ADM2
* Carte du taux de rapportage moyen annuel par ADM2

## 1. Configuration

In [ ]:
# Project paths
SNT_ROOT_PATH <- "/home/hexa/workspace"
PROJECT_PATH <- file.path(SNT_ROOT_PATH, "pipelines/snt_dhis2_reporting_rate_dataelement")
REPORTING_NB_OUTPUTS_PATH <- file.path(PROJECT_PATH, "reporting", "outputs")
CODE_PATH <- file.path(SNT_ROOT_PATH, 'code') # this is where we store snt_utils.r
CONFIG_PATH <- file.path(SNT_ROOT_PATH, 'configuration') # .json config file
DATA_PATH <- file.path(SNT_ROOT_PATH, 'data', 'dhis2')
INTERMEDIATE_RESULTS_PATH <- file.path(DATA_PATH, "reporting_rate", "intermediate_results")

# Load utils
source(file.path(CODE_PATH, "snt_utils.r"))
source(file.path(CODE_PATH, "snt_report.r"))
source(file.path(PROJECT_PATH, "utils", "snt_dhis2_reporting_rate_dataelement_report.r"))

# Load palettes
source(file.path(CODE_PATH, "snt_palettes.r"))

# Load libraries 
required_packages <- c("arrow", "sf", "tidyverse", "stringi", "jsonlite", "httr", "reticulate", "glue", "data.table", "IRdisplay")
install_and_load(required_packages)

# Environment variables
Sys.setenv(PROJ_LIB = "/opt/conda/share/proj")
Sys.setenv(GDAL_DATA = "/opt/conda/share/gdal")
Sys.setenv(RETICULATE_PYTHON = "/opt/conda/bin/python")

# Load OpenHEXA sdk
openhexa <- import("openhexa.sdk")

In [ ]:
# Load SNT config
config_json <- tryCatch({ jsonlite::fromJSON(file.path(CONFIG_PATH, "SNT_config.json")) },
    error = function(e) {
        msg <- paste0("[ERROR] Error while loading configuration", conditionMessage(e))  
        cat(msg)   
        stop(msg) 
    })

pipeline_msg(paste0("SNT configuration loaded from : ", file.path(CONFIG_PATH, "SNT_config.json")))

In [ ]:
# Configuration settings
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE
ADMIN_1 <- toupper(config_json$SNT_CONFIG$DHIS2_ADMINISTRATION_1)
ADMIN_2 <- toupper(config_json$SNT_CONFIG$DHIS2_ADMINISTRATION_2)

# Reporting Rate data is stored in the same OH Dataset regardless of whether is comes from DataSet or DataElement method
REPORTING_RATE_DATASET_NAME <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_REPORTING_RATE
DHIS2_FORMATTED_DATASET_NAME <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_FORMATTED

### Paramètres

Importation des paramètres enregistrés lors de la dernière exécution des calculs.

In [ ]:
# Default values if no parameters file exists
ROUTINE_FILE <- glue("{COUNTRY_CODE}_routine_outliers_imputed.parquet")
DATAELEMENT_METHOD_DENOMINATOR <- "ROUTINE_ACTIVE_FACILITIES" # or "PYRAMID_OPEN_FACILITIES"
ACTIVITY_INDICATORS <- c("CONF", "PRES", "SUSP")
VOLUME_ACTIVITY_INDICATORS <-  c("CONF", "PRES")
USE_WEIGHTED_REPORTING_RATES <- FALSE

In [ ]:
parameters_file <- paste0(COUNTRY_CODE, "_parameters.json")

# Try to load parameters from dataset, if it exists
parameters <- tryCatch({
    get_latest_dataset_file_in_memory(REPORTING_RATE_DATASET_NAME, parameters_file)
}, error = function(e) {
    pipeline_msg(paste0("[WARNING] Parameters could not be loaded; using default values: ", conditionMessage(e)))
    NULL
})

In [ ]:
# Override defaults if parameter list exists and the respective params also exist and are valid
ROUTINE_FILE <- get_param(parameters, "ROUTINE_FILE", ROUTINE_FILE)
DATAELEMENT_METHOD_DENOMINATOR <- get_param(parameters, "DATAELEMENT_METHOD_DENOMINATOR", DATAELEMENT_METHOD_DENOMINATOR, as.character)
ACTIVITY_INDICATORS <- get_param(parameters, "ACTIVITY_INDICATORS", ACTIVITY_INDICATORS)
VOLUME_ACTIVITY_INDICATORS <- get_param(parameters, "VOLUME_ACTIVITY_INDICATORS", VOLUME_ACTIVITY_INDICATORS)
USE_WEIGHTED_REPORTING_RATES <- get_param(parameters, "USE_WEIGHTED_REPORTING_RATES", USE_WEIGHTED_REPORTING_RATES, as.logical)

# Logging what values for the params are used
if (!is.null(parameters) && is.list(parameters)) {
  pipeline_msg(paste0("Parameters loaded (default values if missing) : ", parameters_file))
  pipeline_msg(paste(names(parameters), ": ", parameters))
} else {
  pipeline_msg("Parameter file missing, default values used.")
}

In [ ]:
routine_type <- get_routine_type(input_filename = ROUTINE_FILE)
fosa_denominator <- get_fosa_denominator(input_string = DATAELEMENT_METHOD_DENOMINATOR)

activity_indicators <- paste(ACTIVITY_INDICATORS, collapse = ", ")
volume_activity_indicators <- paste(VOLUME_ACTIVITY_INDICATORS, collapse = ", ")

In [ ]:
# Subtitle text for plots (shows parameter values)
subtitle_text <- paste0(
    "Traitement des cas extrêmes : ", routine_type, 
    "\nFOSA censés rapporter : ", fosa_denominator,
    "\nIndicateurs d'activité : ", activity_indicators,
    "\nIndicateurs de volume : ", volume_activity_indicators,
    "\nPondérations : ", USE_WEIGHTED_REPORTING_RATES
)


In [ ]:
admin_level <- 'ADM2'
admin_id_col <- paste(admin_level, toupper('id'), sep = '_')
admin_name_col <- paste(admin_level, toupper('name'), sep = '_')

year_col <- "YEAR"
month_col <- "MONTH"

data_source = "DHIS2"

num_reporting_categories <- as.integer(5)

In [ ]:
# Load SNT metadata
metadata_json <- tryCatch({ jsonlite::fromJSON(file.path(CONFIG_PATH, "SNT_metadata.json")) },
    error = function(e) {
        msg <- paste0("[ERROR] Error while loading metadata", conditionMessage(e))  
        cat(msg)   
        stop(msg) 
    })

pipeline_msg(paste0("SNT metadata loaded from : ", file.path(CONFIG_PATH, "SNT_metadata.json")))

## 2. Chargement et pré-processing des données à visualiser

**Les données utilisées**

* données administratives : fond de carte au niveau administratif ADM2 (DHIS2)
* données mensuelles et annuelles sur le rapportage, calculées dans le notebook de calcul des taux de rapportage, par ADM2

In [ ]:
# Important: this will break if reporting rate was calculated as DataSet method because it will not find the file
# (will find "{COUNTRY_CODE}_reporting_rate_dataset.parquet" instead)

rr_filename <- glue::glue("{COUNTRY_CODE}_reporting_rate_dataelement.parquet")

monthly_data <- tryCatch({ get_latest_dataset_file_in_memory(REPORTING_RATE_DATASET_NAME, rr_filename) }, 
                  error = function(e) {
                      msg <- paste("Error while loading Reporting Rate (Data Element) data file for: " , COUNTRY_CODE, conditionMessage(e))  # log error message
                      cat(msg)
                      stop(msg)
})

yearly_data <- fread(file.path(INTERMEDIATE_RESULTS_PATH, glue("{COUNTRY_CODE}_reports_dataelement_yearly.csv")))

In [ ]:
# make categories of reporting rates
set.seed(101010)
reporting_k_means_breaks <- make_k_means_breaks(monthly_data[["REPORTING_RATE"]], num_reporting_categories)

# inject into monthly data
monthly_data <- cut_to_categories(monthly_data, "REPORTING_RATE", "REPORTING_RATE_CATEGORY", reporting_k_means_breaks, num_decimals = 2, suffix = "%")

# inject into yearly data
yearly_data <- cut_to_categories(yearly_data, "RR_TOTAL_HF", "REPORTING_RATE_CATEGORY", reporting_k_means_breaks, num_decimals = 2, suffix = "%")

In [ ]:
shapes <- tryCatch({ get_latest_dataset_file_in_memory(DHIS2_FORMATTED_DATASET_NAME, paste0(COUNTRY_CODE, "_shapes.geojson")) }, 
                  error = function(e) {                      
                      msg <- paste0(COUNTRY_CODE , " Shapes data is not available in dataset: '" , DHIS2_FORMATTED_DATASET_NAME, "' last version.")
                      pipeline_msg(msg, "warning")
                      shapes <- NULL
                      })

In [ ]:
monthly_map_data <- monthly_data %>%
  left_join(shapes, by = c("ADM2_ID"))

yearly_map_data <- yearly_data %>%
  left_join(shapes, by = c("ADM1_ID", "ADM1_NAME", "ADM2_ID", "ADM2_NAME"))

## 3. Génération des graphiques de résultat

###  Nuage de points : taux de rapportage en fonction par ADM2 et par mois

Cette visualisation est moins efficace pour identifier les valeurs réelles, mais elle permet de voir si certaines unités administratives présentent des valeurs plus faibles de façon récurrente.

In [ ]:
reporting_rate_scatterplot <- make_reporting_rate_scatterplot(
    df_to_plot = monthly_data,
    admin_colname = admin_id_col,
    year_colname = year_col,
    month_colname = month_col,
    value_colname = "REPORTING_RATE",
    category_colname = "REPORTING_RATE_CATEGORY",
    break_vector = reporting_k_means_breaks,
    plot_palette = cat_reporting_rate_palette,
    none_color = na_color,
    plot_title = "Taux de rapportage\n(Éléments de données)",
    plot_subtitle = subtitle_text,
    plot_caption = glue("Données: {data_source}"),
    x_title = "Mois",
    y_title = "Proportion"
)

In [ ]:
output_file <- paste0(COUNTRY_CODE, "_reporting_rate_dataelement_adm2_linepoint.png")
output_location <- file.path(REPORTING_NB_OUTPUTS_PATH, "figures")

suppressMessages(ggsave(
    filename = output_file, 
    path = output_location,
    plot = reporting_rate_scatterplot,
    create.dir = TRUE,
    height = 15,
    width = 25,
    units = "cm",
    bg = "white",
    dpi = 200
    ))

In [ ]:
display_png(file = file.path(output_location, output_file))

### Carte thermique du taux de rapportage moyen par mois


Cette visualisation est moins efficace pour identifier les valeurs réelles, mais elle permet de voir quels ADM2 présentent des valeurs plus faibles.

In [ ]:
reporting_rate_heatmap <- make_reporting_rate_heatmap(
    df_to_plot = monthly_map_data,
    admin_colname = admin_id_col,
    admin_labels = admin_name_col, 
    year_colname = year_col,
    month_colname = month_col,
    category_colname = "REPORTING_RATE_CATEGORY",
    plot_palette = cat_reporting_rate_palette,
    none_color = na_color,
    plot_title = "Taux de rapportage\n(Éléments de données)",
    plot_subtitle = subtitle_text,
    plot_caption = glue("Données: {data_source}"),
    legend_title = "Proportion",
    x_title = "Mois")


In [ ]:
# save with dynamic height: multiple of the number of rows, plus 3 for the top and bottom text
plot_height <- length(unique(monthly_data[[admin_id_col]])) * 0.22 + 3

output_file <- paste0(COUNTRY_CODE, "_reporting_rate_dataelement_adm2_heatmap.png")
output_location <- file.path(REPORTING_NB_OUTPUTS_PATH, "figures")

suppressMessages(ggsave(
    filename = output_file, 
    path = output_location,
    plot = reporting_rate_heatmap,
    create.dir = TRUE,
    width = 20, height = plot_height, units = "cm",
    limitsize = FALSE,
    dpi = 150
    ))

In [ ]:
display_png(file = file.path(output_location, output_file))

### Carte du taux de rapportage par mois

In [ ]:
monthly_reporting_map <- make_monthly_reporting_map(
    df_to_plot = monthly_map_data,
    admin_colname = admin_id_col,
    year_colname = year_col,
    month_colname = month_col,
    category_colname = "REPORTING_RATE_CATEGORY",
    plot_palette = cat_reporting_rate_palette,
    none_color = na_color,
    plot_title = "Taux de rapportage mensuel\n(Éléments de données)",
    plot_subtitle = subtitle_text,
    plot_caption = glue("Données: {data_source}")
    )

In [ ]:
# save with dynamic width
plot_width <- length(unique(monthly_data[[month_col]])) * 1.5 + 2

output_file <- paste0(COUNTRY_CODE, "_reporting_rate_dataelement_adm2_month_map.png")
output_location <- file.path(REPORTING_NB_OUTPUTS_PATH, "figures")

suppressMessages(ggsave(
    filename = output_file, 
    path = output_location,
    plot = monthly_reporting_map,
    create.dir = TRUE,
    width = plot_width, units = "cm",
    limitsize = FALSE,
    dpi = 150
    ))

In [ ]:
display_png(file = file.path(output_location, output_file))

### Carte du taux de rapportage moyen annuel

In [ ]:
# TODO see if we will ever use weighted yearly data and if yes, adapt this code
yearly_map_subtitle <- paste0("Traitement des cas extrêmes : ", routine_type, 
    "\nFOSA censés rapporter : total \nDonnées non pondérées")

yearly_reporting_map <- make_yearly_reporting_map(
    df_to_plot = yearly_map_data,
    admin_colname = admin_id_col,
    year_colname = year_col,
    category_colname = "REPORTING_RATE_CATEGORY",
    plot_palette = cat_reporting_rate_palette,
    none_color = na_color,
    plot_title = "Taux de rapportage annuel\n(Éléments de données)",
    plot_subtitle = yearly_map_subtitle,
    plot_caption = glue("Données: {data_source}")
    )

In [ ]:
output_file <- paste0(COUNTRY_CODE, "_reporting_rate_dataelement_adm2_year_map.png")
output_location <- file.path(REPORTING_NB_OUTPUTS_PATH, "figures")


suppressMessages(ggsave(
    filename = output_file, 
    path = output_location,
    create.dir = TRUE, 
    width = 25, height = 10, units = "cm", 
    dpi = 200
    ))


In [ ]:
display_png(file = file.path(output_location, output_file))

In [ ]:
pipeline_msg("Reporting Rate (Data Element) report notebook completed successfully!")